In [3]:
# ============================================================
# Starts from:
#   06_combined_tfg_respicast_predictions_long_corrected.csv
#
# Main metrics:
#   - mean_AE
#   - mean_scaled_AE
#   - RMSE
#
# Main table:
#   for each scope / disease / horizon:
#     - top 3 TFG methods
#     - top 2 RespiCast methods excluding baseline
#     - respicast-quantileBaseline
# ============================================================

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]

REFERENCE_MODEL = "respicast-quantileBaseline"
REFERENCE_METHOD = f"RespiCast::{REFERENCE_MODEL}"

HORIZONS = [1, 2, 3, 4]

TOP_TFG_N = 3
TOP_RESPICAST_N = 2
MIN_COVERAGE_VS_BASELINE_FOR_MAIN = 0.50

# Incidence is a rate per 100,000, so values above this are impossible.
# They are excluded from the main simplified table and audited separately.
EXTREME_UPPER_BOUND = 100000.0

SCALE_START = pd.Timestamp("2014-01-05")
SCALE_END = pd.Timestamp("2023-12-31")
WEEK_FREQ = "W-SUN"
M_SEASON = 52

# ------------------------------------------------------------
# 1) PATHS
# ------------------------------------------------------------
def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]
    for c in candidates:
        if (c / "results").exists() and (c / "data").exists():
            return c.resolve()
    raise FileNotFoundError("Could not find project root containing 'results' and 'data' folders.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FINAL_TEST_DIR = RESULTS_DIR / "final_test_2024_2025"

preferred_combined = (
    FINAL_TEST_DIR
    / "respicast_comparison_corrected"
    / "06_combined_tfg_respicast_predictions_long_corrected.csv"
)

if preferred_combined.exists():
    COMBINED_PATH = preferred_combined
else:
    matches = list(RESULTS_DIR.rglob("06_combined_tfg_respicast_predictions_long_corrected.csv"))
    if not matches:
        raise FileNotFoundError(
            "Could not find 06_combined_tfg_respicast_predictions_long_corrected.csv. "
            "Run the corrected RespiCast comparison notebook first."
        )
    COMBINED_PATH = matches[0]

OUT_DIR = FINAL_TEST_DIR / "respicast_final_clean_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("COMBINED_PATH:", COMBINED_PATH)
print("OUT_DIR:", OUT_DIR)

# ------------------------------------------------------------
# 2) IMPORT SRC FUNCTIONS FOR SCALE COMPUTATION
# ------------------------------------------------------------
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.impute import impute_series_weekly
    from src.metrics import mase_scale_seasonal
    SRC_OK = True
    print("src functions loaded.")
except Exception as e:
    SRC_OK = False
    print("WARNING: src functions not available. Fallback scale computation will be used.")
    print("Error:", e)

# ------------------------------------------------------------
# 3) HELPERS
# ------------------------------------------------------------
def normalize_columns(df):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

def find_incidence_file(disease):
    """
    Robustly find latest ARI/ILI incidence files, regardless of case.
    """
    disease = disease.upper()
    candidates = [
        DATA_DIR / f"latest-{disease}_incidence.csv",
        DATA_DIR / f"latest-{disease.lower()}_incidence.csv",
        DATA_DIR / f"latest_{disease}_incidence.csv",
        DATA_DIR / f"latest_{disease.lower()}_incidence.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    all_csv = list(DATA_DIR.glob("*.csv"))
    matches = [
        p for p in all_csv
        if "latest" in p.name.lower()
        and disease.lower() in p.name.lower()
        and "incidence" in p.name.lower()
    ]

    if matches:
        return sorted(matches)[-1]

    matches = [
        p for p in all_csv
        if disease.lower() in p.name.lower()
        and "incidence" in p.name.lower()
    ]

    if matches:
        return sorted(matches)[-1]

    raise FileNotFoundError(f"No incidence file found for {disease} in {DATA_DIR}")

def read_incidence_file(disease):
    p = find_incidence_file(disease)
    df = pd.read_csv(p)
    df = normalize_columns(df)

    required = {"truth_date", "location", "value"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Incidence file {p} is missing columns: {missing}")

    df["truth_date"] = pd.to_datetime(df["truth_date"], errors="coerce")
    df["location"] = df["location"].astype(str)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = (
        df.dropna(subset=["truth_date", "location", "value"])
        .drop_duplicates(["location", "truth_date"], keep="last")
        .sort_values(["location", "truth_date"])
        .reset_index(drop=True)
    )

    print(f"{disease} incidence file:", p.name, "| shape:", df.shape)
    return df

def compute_scale_for_location(df_hist, location):
    """
    Seasonal MASE scale computed on pre-final-test historical data.
    Same idea as the rest of the TFG:
    mean absolute seasonal difference with m=52.
    """
    calendar = pd.date_range(SCALE_START, SCALE_END, freq=WEEK_FREQ)

    dfloc = df_hist[df_hist["location"] == location][["truth_date", "value"]].copy()
    dfloc["truth_date"] = pd.to_datetime(dfloc["truth_date"], errors="coerce")
    dfloc["value"] = pd.to_numeric(dfloc["value"], errors="coerce")
    dfloc = dfloc.dropna(subset=["truth_date", "value"])

    if dfloc.empty:
        return np.nan

    if SRC_OK:
        s = impute_series_weekly(
            dfloc,
            calendar=calendar,
            trim_to_first_obs=True,
            max_gap_knn=8
        )
        scale = float(mase_scale_seasonal(s, m=M_SEASON))
    else:
        s = (
            dfloc.sort_values("truth_date")
            .drop_duplicates("truth_date", keep="last")
            .set_index("truth_date")["value"]
            .reindex(calendar)
            .astype(float)
        )
        s = s.interpolate(limit_direction="both").ffill().bfill()
        scale = float((s - s.shift(M_SEASON)).abs().dropna().mean())

    if not np.isfinite(scale) or scale <= 0:
        return np.nan

    return scale

def build_scales_for_eval(df_eval):
    ari_hist = read_incidence_file("ARI")
    ili_hist = read_incidence_file("ILI")

    rows = []

    for disease, hist in [("ARI", ari_hist), ("ILI", ili_hist)]:
        locs = sorted(df_eval[df_eval["disease"] == disease]["location"].unique())

        for loc in locs:
            try:
                scale = compute_scale_for_location(hist, loc)
            except Exception as e:
                print(f"WARNING: failed scale for {disease}-{loc}: {e}")
                scale = np.nan

            rows.append({
                "disease": disease,
                "location": loc,
                "mase_scale": scale
            })

    scales = pd.DataFrame(rows)

    if scales.empty:
        raise RuntimeError("Scale table is empty. Cannot compute scaled_AE.")

    if scales["mase_scale"].isna().any():
        bad = scales[scales["mase_scale"].isna()].copy()
        print("\nWARNING: missing mase_scale for these disease-location pairs:")
        display(bad)

    return scales

def add_scope(df, scope_name):
    out = df.copy()
    out["scope"] = scope_name
    return out

def safe_ratio(num, den):
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return num / den

# ------------------------------------------------------------
# 4) LOAD COMBINED CORRECTED PREDICTIONS
# ------------------------------------------------------------
df = pd.read_csv(COMBINED_PATH)
df = normalize_columns(df)

required = {"origin", "target", "disease", "location", "h", "y", "pred", "model", "source"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in combined corrected file: {missing}")

if "method" not in df.columns:
    df["method"] = df["source"].astype(str) + "::" + df["model"].astype(str)

df["origin"] = pd.to_datetime(df["origin"], errors="coerce")
df["target"] = pd.to_datetime(df["target"], errors="coerce")
df["disease"] = df["disease"].astype(str).str.upper()
df["location"] = df["location"].astype(str)
df["h"] = pd.to_numeric(df["h"], errors="coerce")
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df["pred"] = pd.to_numeric(df["pred"], errors="coerce")
df["model"] = df["model"].astype(str)
df["source"] = df["source"].astype(str)
df["method"] = df["method"].astype(str)

original_rows = len(df)

df = df.dropna(
    subset=["origin", "target", "disease", "location", "h", "y", "model", "source", "method"]
).copy()

df = df[np.isfinite(df["y"])].copy()

# Date-based horizon is the final horizon definition.
df["h_from_dates"] = ((df["target"] - df["origin"]).dt.days / 7).round(6)

horizon_date_audit = (
    df.groupby(["source", "model", "h", "h_from_dates"], as_index=False)
    .size()
    .rename(columns={"size": "n_rows"})
    .sort_values(["source", "model", "h", "h_from_dates"])
)

df = df[df["h_from_dates"].isin(HORIZONS)].copy()
df["h_original_column"] = df["h"]
df["h"] = df["h_from_dates"].astype(int)
df = df[df["h"].isin(HORIZONS)].copy()

print("\nOriginal rows:", original_rows)
print("Rows after key/truth/date-horizon cleaning:", len(df))
print("\nHorizon/date audit preview:")
display(horizon_date_audit.head(50))

# ------------------------------------------------------------
# 5) PREDICTION QUALITY AUDITS
# ------------------------------------------------------------
df["pred_is_finite"] = np.isfinite(df["pred"])
df["pred_is_negative"] = df["pred_is_finite"] & (df["pred"] < 0)
df["pred_is_extreme_high"] = df["pred_is_finite"] & (df["pred"] > EXTREME_UPPER_BOUND)

df["pred_quality_flag"] = np.select(
    [
        ~df["pred_is_finite"],
        df["pred_is_negative"],
        df["pred_is_extreme_high"]
    ],
    [
        "non_finite_prediction",
        "negative_prediction",
        "above_100000_per_100000"
    ],
    default="finite_non_negative_plausible"
)

prediction_quality_audit = (
    df.groupby(["source", "model", "method", "disease", "h", "pred_quality_flag"], as_index=False)
    .agg(
        n_rows=("pred", "size"),
        min_pred=("pred", "min"),
        max_pred=("pred", "max"),
        mean_pred=("pred", "mean"),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max")
    )
    .sort_values(["pred_quality_flag", "source", "model", "disease", "h"])
)

invalid_or_extreme_rows = df[df["pred_quality_flag"] != "finite_non_negative_plausible"].copy()

print("\nPrediction quality audit:")
display(prediction_quality_audit)

print("\nInvalid/extreme rows:", invalid_or_extreme_rows.shape)
if len(invalid_or_extreme_rows) > 0:
    display(invalid_or_extreme_rows.head(20))

# Main evaluation:
# - non-finite predictions cannot be evaluated
# - extremely high impossible predictions are excluded from main comparison and audited
# - negative finite predictions are kept and penalised by their error
df_eval = df[df["pred_is_finite"] & (~df["pred_is_extreme_high"])].copy()

# ------------------------------------------------------------
# 6) DEDUPLICATE EXACT FORECAST KEYS IF NECESSARY
# ------------------------------------------------------------
PRED_KEYS = ["source", "model", "method", "disease", "location", "origin", "target", "h"]

duplicate_rows = df_eval[df_eval.duplicated(PRED_KEYS, keep=False)].copy()

if len(duplicate_rows) > 0:
    print("\nWARNING: duplicated prediction keys found. Aggregating predictions by mean.")
    display(duplicate_rows.head(20))

    duplicate_audit = (
        duplicate_rows.groupby(PRED_KEYS, as_index=False)
        .agg(
            n_duplicates=("pred", "size"),
            min_pred=("pred", "min"),
            max_pred=("pred", "max"),
            mean_pred=("pred", "mean"),
            y_min=("y", "min"),
            y_max=("y", "max")
        )
    )
else:
    duplicate_audit = pd.DataFrame(columns=PRED_KEYS + ["n_duplicates", "min_pred", "max_pred", "mean_pred", "y_min", "y_max"])

# Aggregate only if duplicates exist. y should be identical for duplicate forecast keys.
df_eval = (
    df_eval.groupby(PRED_KEYS, as_index=False)
    .agg(
        y=("y", "mean"),
        pred=("pred", "mean"),
        pred_is_negative=("pred_is_negative", "max"),
        pred_is_extreme_high=("pred_is_extreme_high", "max"),
        pred_quality_flag=("pred_quality_flag", lambda s: "|".join(sorted(set(map(str, s)))))
    )
)

# ------------------------------------------------------------
# 7) ADD SCOPES
# ------------------------------------------------------------
scoped = pd.concat(
    [
        add_scope(df_eval, "global"),
        add_scope(df_eval[df_eval["location"].isin(COMMON6)], "common6")
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 8) ADD MASE SCALES ROBUSTLY
# ------------------------------------------------------------
scales = build_scales_for_eval(scoped)

scale_like_cols = [c for c in scoped.columns if c.lower().startswith("mase_scale")]
if scale_like_cols:
    print("Dropping existing scale-like columns before merge:", scale_like_cols)
    scoped = scoped.drop(columns=scale_like_cols)

scoped = scoped.merge(
    scales,
    on=["disease", "location"],
    how="left",
    validate="many_to_one"
)

if "mase_scale" not in scoped.columns:
    raise RuntimeError(
        "mase_scale was not created after merge. Columns are: "
        + ", ".join(scoped.columns)
    )

if scoped["mase_scale"].isna().any():
    bad_scale_rows = scoped[scoped["mase_scale"].isna()][["disease", "location"]].drop_duplicates()
    print("\nERROR: missing mase_scale for these pairs:")
    display(bad_scale_rows)
    raise RuntimeError("Missing mase_scale values. Fix scales before evaluation.")

scoped["AE"] = (scoped["y"] - scoped["pred"]).abs()
scoped["SE"] = (scoped["y"] - scoped["pred"]) ** 2
scoped["scaled_AE"] = scoped["AE"] / scoped["mase_scale"]

# ------------------------------------------------------------
# 9) PAIR EACH METHOD WITH RESPICAST BASELINE
# ------------------------------------------------------------
ref = scoped[scoped["method"] == REFERENCE_METHOD].copy()

if ref.empty:
    raise ValueError(
        f"No reference rows found for {REFERENCE_METHOD}. "
        "Check that respicast-quantileBaseline exists in the combined corrected file."
    )

PAIR_KEYS = ["scope", "disease", "location", "origin", "target", "h"]

ref_dup = ref.duplicated(PAIR_KEYS).sum()
if ref_dup > 0:
    print(f"\nWARNING: baseline has {ref_dup} duplicated keys after cleaning. Aggregating by mean.")
    ref = (
        ref.groupby(PAIR_KEYS, as_index=False)
        .agg(
            y=("y", "mean"),
            pred=("pred", "mean"),
            AE=("AE", "mean"),
            SE=("SE", "mean"),
            scaled_AE=("scaled_AE", "mean")
        )
    )

ref_for_pairing = ref[
    PAIR_KEYS + ["y", "pred", "AE", "SE", "scaled_AE"]
].rename(
    columns={
        "y": "y_ref",
        "pred": "pred_ref",
        "AE": "AE_ref",
        "SE": "SE_ref",
        "scaled_AE": "scaled_AE_ref"
    }
)

cand_for_pairing = scoped[
    PAIR_KEYS + [
        "source", "model", "method", "y", "pred",
        "AE", "SE", "scaled_AE",
        "pred_is_negative", "pred_quality_flag"
    ]
].rename(
    columns={
        "y": "y_model",
        "pred": "pred_model",
        "AE": "AE_model",
        "SE": "SE_model",
        "scaled_AE": "scaled_AE_model"
    }
)

paired = cand_for_pairing.merge(ref_for_pairing, on=PAIR_KEYS, how="inner")

paired["truth_abs_diff_vs_ref"] = (paired["y_model"] - paired["y_ref"]).abs()
max_truth_diff = paired["truth_abs_diff_vs_ref"].max()

print("\nPaired rows:", len(paired))
print("Max truth difference after pairing:", max_truth_diff)

if max_truth_diff > 1e-8:
    raise ValueError("Truth mismatch after pairing with baseline. Evaluation is not safe.")

# ------------------------------------------------------------
# 10) ZERO ERROR AUDIT
# ------------------------------------------------------------
paired["model_AE_zero"] = np.isclose(paired["AE_model"], 0.0)
paired["ref_AE_zero"] = np.isclose(paired["AE_ref"], 0.0)
paired["both_AE_zero"] = paired["model_AE_zero"] & paired["ref_AE_zero"]
paired["y_zero"] = np.isclose(paired["y_model"], 0.0)
paired["pred_model_zero"] = np.isclose(paired["pred_model"], 0.0)
paired["pred_ref_zero"] = np.isclose(paired["pred_ref"], 0.0)

zero_error_audit = (
    paired.groupby(["scope", "source", "model", "method", "disease", "h"], as_index=False)
    .agg(
        n_pairs=("AE_model", "size"),
        n_y_zero=("y_zero", "sum"),
        n_pred_model_zero=("pred_model_zero", "sum"),
        n_pred_ref_zero=("pred_ref_zero", "sum"),
        n_model_AE_zero=("model_AE_zero", "sum"),
        n_ref_AE_zero=("ref_AE_zero", "sum"),
        n_both_AE_zero=("both_AE_zero", "sum"),
        mean_AE_model=("AE_model", "mean"),
        mean_AE_ref=("AE_ref", "mean")
    )
    .sort_values(["scope", "disease", "h", "source", "model"])
)

# ------------------------------------------------------------
# 11) METRIC SUMMARY
# ------------------------------------------------------------
reference_universe = (
    paired[paired["method"] == REFERENCE_METHOD]
    .groupby(["scope", "disease", "h"], as_index=False)
    .agg(
        reference_pairs=("AE_model", "size"),
        reference_countries=("location", "nunique"),
        reference_first_origin=("origin", "min"),
        reference_last_origin=("origin", "max"),
        reference_first_target=("target", "min"),
        reference_last_target=("target", "max")
    )
)

metric_summary = (
    paired.groupby(["scope", "source", "model", "method", "disease", "h"], as_index=False)
    .agg(
        mean_AE=("AE_model", "mean"),
        median_AE=("AE_model", "median"),
        RMSE=("SE_model", lambda x: float(np.sqrt(np.mean(x)))),
        mean_scaled_AE=("scaled_AE_model", "mean"),
        median_scaled_AE=("scaled_AE_model", "median"),
        mean_AE_ref_baseline=("AE_ref", "mean"),
        mean_scaled_AE_ref_baseline=("scaled_AE_ref", "mean"),
        n_pairs=("AE_model", "size"),
        n_countries=("location", "nunique"),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max"),
        n_negative_predictions=("pred_is_negative", "sum")
    )
)

metric_summary = metric_summary.merge(
    reference_universe,
    on=["scope", "disease", "h"],
    how="left",
    validate="many_to_one"
)

metric_summary["coverage_vs_baseline"] = metric_summary["n_pairs"] / metric_summary["reference_pairs"]

metric_summary["MAE_ratio_vs_baseline"] = metric_summary.apply(
    lambda r: safe_ratio(r["mean_AE"], r["mean_AE_ref_baseline"]),
    axis=1
)

metric_summary["scaled_AE_ratio_vs_baseline"] = metric_summary.apply(
    lambda r: safe_ratio(r["mean_scaled_AE"], r["mean_scaled_AE_ref_baseline"]),
    axis=1
)

metric_summary["rank_by_scaled_AE"] = (
    metric_summary.groupby(["scope", "disease", "h"])["mean_scaled_AE"]
    .rank(method="min", ascending=True)
    .astype(int)
)

metric_summary["rank_by_AE"] = (
    metric_summary.groupby(["scope", "disease", "h"])["mean_AE"]
    .rank(method="min", ascending=True)
    .astype(int)
)

metric_summary = metric_summary.sort_values(
    ["scope", "disease", "h", "mean_scaled_AE", "mean_AE"]
).reset_index(drop=True)

print("\nMetric summary preview:")
display(metric_summary.head(50))

# ------------------------------------------------------------
# 12) MAIN SIMPLIFIED TABLE
# ------------------------------------------------------------
main_parts = []

for (scope, disease, h), sub in metric_summary.groupby(["scope", "disease", "h"]):
    sub = sub.copy()

    tfg_top = (
        sub[sub["source"] == "TFG"]
        .sort_values(["mean_scaled_AE", "mean_AE"])
        .head(TOP_TFG_N)
    )

    respicast_top = (
        sub[
            (sub["source"] == "RespiCast")
            & (sub["method"] != REFERENCE_METHOD)
            & (sub["coverage_vs_baseline"] >= MIN_COVERAGE_VS_BASELINE_FOR_MAIN)
        ]
        .sort_values(["mean_scaled_AE", "mean_AE"])
        .head(TOP_RESPICAST_N)
    )

    baseline = sub[sub["method"] == REFERENCE_METHOD]

    selected = pd.concat([tfg_top, respicast_top, baseline], ignore_index=True)
    selected = selected.drop_duplicates(subset=["scope", "disease", "h", "method"])
    selected = selected.sort_values(["mean_scaled_AE", "mean_AE"]).reset_index(drop=True)
    selected["main_table_rank"] = np.arange(1, len(selected) + 1)

    main_parts.append(selected)

main_table = pd.concat(main_parts, ignore_index=True)

main_cols = [
    "scope",
    "disease",
    "h",
    "main_table_rank",
    "source",
    "model",
    "mean_AE",
    "mean_scaled_AE",
    "RMSE",
    "median_AE",
    "n_pairs",
    "reference_pairs",
    "coverage_vs_baseline",
    "n_countries",
    "mean_AE_ref_baseline",
    "MAE_ratio_vs_baseline",
    "scaled_AE_ratio_vs_baseline",
    "n_negative_predictions",
    "first_origin",
    "last_origin",
    "first_target",
    "last_target"
]

main_table = main_table[main_cols].copy()

print("\nMAIN TABLE:")
display(main_table)

# ------------------------------------------------------------
# 13) AUXILIARY TABLES
# ------------------------------------------------------------
top10_all = (
    metric_summary.sort_values(["scope", "disease", "h", "mean_scaled_AE", "mean_AE"])
    .groupby(["scope", "disease", "h"], group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

top10_coverage_filtered = (
    metric_summary[
        (metric_summary["source"] == "TFG")
        | (metric_summary["coverage_vs_baseline"] >= MIN_COVERAGE_VS_BASELINE_FOR_MAIN)
    ]
    .sort_values(["scope", "disease", "h", "mean_scaled_AE", "mean_AE"])
    .groupby(["scope", "disease", "h"], group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

tfg_positions = (
    metric_summary[metric_summary["source"] == "TFG"]
    .sort_values(["scope", "disease", "h", "mean_scaled_AE", "mean_AE"])
    .reset_index(drop=True)
)

respicast_positions = (
    metric_summary[metric_summary["source"] == "RespiCast"]
    .sort_values(["scope", "disease", "h", "mean_scaled_AE", "mean_AE"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 14) COLUMN DICTIONARY
# ------------------------------------------------------------
column_dictionary = pd.DataFrame([
    {
        "column": "scope",
        "definition": "Evaluation universe. global = all countries available for the TFG final evaluation for each disease; common6 = BE, CZ, EE, LT, RO and SI."
    },
    {
        "column": "disease",
        "definition": "Respiratory indicator: ARI or ILI."
    },
    {
        "column": "h",
        "definition": "Forecast horizon in weeks. h=1 means target = origin + 1 week."
    },
    {
        "column": "main_table_rank",
        "definition": "Rank inside each scope-disease-horizon block in the simplified table. Lower is better."
    },
    {
        "column": "source",
        "definition": "Origin of the method: TFG or RespiCast."
    },
    {
        "column": "model",
        "definition": "Model name as stored in the prediction files."
    },
    {
        "column": "mean_AE",
        "definition": "Mean absolute error of the model on the matched forecast-observation pairs."
    },
    {
        "column": "mean_scaled_AE",
        "definition": "Mean absolute error divided by the seasonal naive in-sample scale for the corresponding disease and country."
    },
    {
        "column": "RMSE",
        "definition": "Root mean squared error on the matched forecast-observation pairs."
    },
    {
        "column": "median_AE",
        "definition": "Median absolute error on the matched forecast-observation pairs."
    },
    {
        "column": "n_pairs",
        "definition": "Number of matched forecast-observation pairs used for that method."
    },
    {
        "column": "reference_pairs",
        "definition": "Number of available respicast-quantileBaseline pairs for the same scope, disease and horizon."
    },
    {
        "column": "coverage_vs_baseline",
        "definition": "n_pairs divided by reference_pairs. It measures the method's overlap with the RespiCast baseline."
    },
    {
        "column": "n_countries",
        "definition": "Number of countries represented in the matched comparison."
    },
    {
        "column": "mean_AE_ref_baseline",
        "definition": "Mean AE of respicast-quantileBaseline on exactly the same matched pairs as the evaluated method."
    },
    {
        "column": "MAE_ratio_vs_baseline",
        "definition": "mean_AE / mean_AE_ref_baseline. Values below 1 mean the method has lower MAE than the baseline on the same matched pairs."
    },
    {
        "column": "scaled_AE_ratio_vs_baseline",
        "definition": "mean_scaled_AE / mean_scaled_AE_ref_baseline. Values below 1 mean the method has lower scaled AE than the baseline on the same matched pairs."
    },
    {
        "column": "n_negative_predictions",
        "definition": "Number of negative finite predictions. They are kept in the evaluation and penalised through their error, but audited because incidence cannot be negative."
    }
])

# ------------------------------------------------------------
# 15) METHODOLOGY NOTES
# ------------------------------------------------------------
methodology_notes = f"""
Final RespiCast comparison methodology
======================================

Input
-----
This notebook starts from:
{COMBINED_PATH}

Forecast alignment
------------------
The evaluation keeps only rows where:
target = origin + h weeks, with h in {{1, 2, 3, 4}}.

The final horizon used in the tables is computed from the dates, not only from
the raw horizon column.

Reference model
---------------
The RespiCast reference model is:
{REFERENCE_METHOD}

Pairing
-------
Each method is compared only on forecast-observation pairs that are also
available for respicast-quantileBaseline. The matching keys are:
scope, disease, location, origin, target and h.

Main metrics
------------
The main comparison uses:
- mean_AE
- mean_scaled_AE
- RMSE

The primary ranking in the simplified table is by mean_scaled_AE and then mean_AE.

Why log2 is not used as the main metric
---------------------------------------
Pointwise log2(AE_ref / AE_model) is problematic when individual absolute errors
are zero. Zero errors can occur legitimately when incidence is very low or zero.

The aggregate log2(mean_AE_ref / mean_AE_model) is also not used as the main
criterion because it gives the same ordering as mean_AE on the same paired sample.

Therefore, the final tables use direct MAE and scaled AE metrics.

Zero errors
-----------
Zero absolute errors are valid. They are not removed. They are included in MAE
and scaled AE. They are reported separately in the zero-error audit.

Prediction quality
------------------
Non-finite predictions mean NaN, +inf or -inf. These cannot be evaluated and are
excluded.
Extremely high forecasts above {EXTREME_UPPER_BOUND:,.0f} per 100,000 are excluded
from the main comparison and reported in the quality audit.
Negative finite forecasts are kept in the evaluation and penalised by their error,
but they are audited separately because incidence cannot be negative.

Simplified main table
---------------------
For each scope, disease and horizon, the simplified main table contains:
- the top {TOP_TFG_N} TFG methods,
- the top {TOP_RESPICAST_N} RespiCast methods excluding the baseline,
- respicast-quantileBaseline.

For RespiCast competitors, the table only includes methods with
coverage_vs_baseline >= {MIN_COVERAGE_VS_BASELINE_FOR_MAIN}.
"""

# ------------------------------------------------------------
# 16) SAVE OUTPUTS
# ------------------------------------------------------------
outputs = {
    "00_column_dictionary.csv": column_dictionary,
    "01_horizon_date_audit.csv": horizon_date_audit,
    "02_prediction_quality_audit.csv": prediction_quality_audit,
    "03_invalid_or_extreme_prediction_rows.csv": invalid_or_extreme_rows,
    "04_duplicate_prediction_key_audit.csv": duplicate_audit,
    "05_scales_used.csv": scales,
    "06_paired_forecasts_vs_baseline.csv": paired,
    "07_zero_error_audit.csv": zero_error_audit,
    "08_metric_summary_all_methods.csv": metric_summary,
    "09_MAIN_top3_TFG_vs_top2_RespiCast_plus_baseline.csv": main_table,
    "10_top10_all_methods_by_scope_disease_horizon.csv": top10_all,
    "11_top10_coverage_filtered.csv": top10_coverage_filtered,
    "12_tfg_positions_only.csv": tfg_positions,
    "13_respicast_positions_only.csv": respicast_positions,
    "14_reference_universe.csv": reference_universe
}

for fname, table in outputs.items():
    table.to_csv(OUT_DIR / fname, index=False)

with open(OUT_DIR / "00_methodology_notes.txt", "w", encoding="utf-8") as f:
    f.write(methodology_notes)

print("\nSaved final clean RespiCast comparison outputs to:")
print(OUT_DIR)

print("\nGenerated files:")
for fname in outputs:
    print("-", fname)
print("- 00_methodology_notes.txt")

print("\n============================================================")
print("MAIN FINAL TABLE")
print("============================================================")
display(main_table)

print("\n============================================================")
print("TFG POSITIONS")
print("============================================================")
display(tfg_positions)

print("\n============================================================")
print("ZERO ERROR AUDIT")
print("============================================================")
display(zero_error_audit)

print("\n============================================================")
print("PREDICTION QUALITY AUDIT")
print("============================================================")
display(prediction_quality_audit)

print("\n============================================================")
print("SCALES USED")
print("============================================================")
display(scales)

PROJECT_ROOT: C:\Users\aolas\UNI PYTHON\TFG
DATA_DIR: C:\Users\aolas\UNI PYTHON\TFG\data
RESULTS_DIR: C:\Users\aolas\UNI PYTHON\TFG\results
COMBINED_PATH: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\respicast_comparison_corrected\06_combined_tfg_respicast_predictions_long_corrected.csv
OUT_DIR: C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\respicast_final_clean_tables
src functions loaded.

Original rows: 92961
Rows after key/truth/date-horizon cleaning: 92961

Horizon/date audit preview:


,source,model,h,h_from_dates,n_rows
0,RespiCast,ECDC-SARIMA,1,1.0,927
1,RespiCast,ECDC-SARIMA,2,2.0,907
2,RespiCast,ECDC-SARIMA,3,3.0,894
3,RespiCast,ECDC-SARIMA,4,4.0,878
4,RespiCast,ECDC-soca_simplex,1,1.0,1021
5,RespiCast,ECDC-soca_simplex,2,2.0,996
6,RespiCast,ECDC-soca_simplex,3,3.0,967
7,RespiCast,ECDC-soca_simplex,4,4.0,940
8,RespiCast,ISI-FluABCaster,1,1.0,464
9,RespiCast,ISI-FluABCaster,2,2.0,444



Prediction quality audit:


,source,model,method,disease,h,pred_quality_flag,n_rows,min_pred,max_pred,mean_pred,first_origin,last_origin,first_target,last_target
112,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,1,above_100000_per_100000,1,2.842952e+08,2.842952e+08,2.842952e+08,2025-06-15,2025-06-15,2025-06-22,2025-06-22
114,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,2,above_100000_per_100000,1,5.725977e+15,5.725977e+15,5.725977e+15,2025-06-15,2025-06-15,2025-06-29,2025-06-29
116,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,3,above_100000_per_100000,1,1.153267e+23,1.153267e+23,1.153267e+23,2025-06-15,2025-06-15,2025-07-06,2025-07-06
118,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,4,above_100000_per_100000,1,2.322790e+30,2.322790e+30,2.322790e+30,2025-06-15,2025-06-15,2025-07-13,2025-07-13
0,RespiCast,ECDC-SARIMA,RespiCast::ECDC-SARIMA,ARI,1,finite_non_negative_plausible,407,9.150812e+00,2.828806e+03,8.908229e+02,2024-10-13,2025-12-07,2024-10-20,2025-12-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,TFG,autoARIMA,TFG::autoARIMA,ILI,2,negative_prediction,33,-1.580170e+01,-2.948716e-98,-2.809473e+00,2024-03-10,2025-11-09,2024-03-24,2025-11-23
219,TFG,autoARIMA,TFG::autoARIMA,ILI,3,negative_prediction,46,-2.740079e+01,-2.603163e-98,-3.916548e+00,2024-03-10,2025-11-09,2024-03-31,2025-11-30
221,TFG,autoARIMA,TFG::autoARIMA,ILI,4,negative_prediction,56,-2.709863e+01,-1.961124e-98,-4.294071e+00,2024-01-07,2025-11-09,2024-02-04,2025-12-07
227,TFG,ensemble_mean_5,TFG::ensemble_mean_5,ILI,1,negative_prediction,1,-2.853516e-01,-2.853516e-01,-2.853516e-01,2025-05-11,2025-05-11,2025-05-18,2025-05-18



Invalid/extreme rows: (352, 27)


...

,scope,source,model,method,disease,h,n_pairs,n_y_zero,n_pred_model_zero,n_pred_ref_zero,n_model_AE_zero,n_ref_AE_zero,n_both_AE_zero,mean_AE_model,mean_AE_ref
0,common6,RespiCast,ECDC-SARIMA,RespiCast::ECDC-SARIMA,ARI,1,290,1,0,0,0,0,0,131.683221,125.687069
8,common6,RespiCast,ECDC-soca_simplex,RespiCast::ECDC-soca_simplex,ARI,1,339,1,0,0,0,0,0,140.517958,132.866224
16,common6,RespiCast,ISI-FluABCaster,RespiCast::ISI-FluABCaster,ARI,1,145,1,0,0,0,0,0,182.201256,160.358276
24,common6,RespiCast,ISI-FluBcast,RespiCast::ISI-FluBcast,ARI,1,232,1,0,0,0,0,0,133.142850,138.791379
32,common6,RespiCast,ISI-GLEAM,RespiCast::ISI-GLEAM,ARI,1,112,1,0,0,0,0,0,251.690536,195.095982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,global,TFG,"SARIMA(1,0,0)x(1,0,0)[52] (initial)","TFG::SARIMA(1,0,0)x(1,0,0)[52] (initial)",ILI,4,572,23,0,13,0,12,0,59.833497,70.959091
431,global,TFG,autoARIMA,TFG::autoARIMA,ILI,4,572,23,19,13,12,12,7,61.780247,70.959091
439,global,TFG,ensemble_mean_5,TFG::ensemble_mean_5,ILI,4,572,23,0,13,0,12,0,50.456452,70.959091
447,global,TFG,ensemble_median_5,TFG::ensemble_median_5,ILI,4,572,23,0,13,0,12,0,53.069211,70.959091



PREDICTION QUALITY AUDIT


,source,model,method,disease,h,pred_quality_flag,n_rows,min_pred,max_pred,mean_pred,first_origin,last_origin,first_target,last_target
112,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,1,above_100000_per_100000,1,2.842952e+08,2.842952e+08,2.842952e+08,2025-06-15,2025-06-15,2025-06-22,2025-06-22
114,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,2,above_100000_per_100000,1,5.725977e+15,5.725977e+15,5.725977e+15,2025-06-15,2025-06-15,2025-06-29,2025-06-29
116,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,3,above_100000_per_100000,1,1.153267e+23,1.153267e+23,1.153267e+23,2025-06-15,2025-06-15,2025-07-06,2025-07-06
118,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,4,above_100000_per_100000,1,2.322790e+30,2.322790e+30,2.322790e+30,2025-06-15,2025-06-15,2025-07-13,2025-07-13
0,RespiCast,ECDC-SARIMA,RespiCast::ECDC-SARIMA,ARI,1,finite_non_negative_plausible,407,9.150812e+00,2.828806e+03,8.908229e+02,2024-10-13,2025-12-07,2024-10-20,2025-12-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,TFG,autoARIMA,TFG::autoARIMA,ILI,2,negative_prediction,33,-1.580170e+01,-2.948716e-98,-2.809473e+00,2024-03-10,2025-11-09,2024-03-24,2025-11-23
219,TFG,autoARIMA,TFG::autoARIMA,ILI,3,negative_prediction,46,-2.740079e+01,-2.603163e-98,-3.916548e+00,2024-03-10,2025-11-09,2024-03-31,2025-11-30
221,TFG,autoARIMA,TFG::autoARIMA,ILI,4,negative_prediction,56,-2.709863e+01,-1.961124e-98,-4.294071e+00,2024-01-07,2025-11-09,2024-02-04,2025-12-07
227,TFG,ensemble_mean_5,TFG::ensemble_mean_5,ILI,1,negative_prediction,1,-2.853516e-01,-2.853516e-01,-2.853516e-01,2025-05-11,2025-05-11,2025-05-18,2025-05-18



SCALES USED


,disease,location,mase_scale
0,ARI,BE,299.077262
1,ARI,BG,184.456148
2,ARI,CZ,192.408585
3,ARI,DE,325.907657
4,ARI,EE,79.899768
5,ARI,LT,160.480639
6,ARI,RO,138.173318
7,ARI,SI,430.407657
8,ILI,BE,112.301160
9,ILI,CZ,20.550696
